# Crime Arrest Prediction – Modeling Pipeline

Predicts whether a crime results in an arrest.

**Random Forest Regressor**


## 1. Imports

In [41]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix


## 2. Load Data

Dataset columns:

- primary_type
- district
- hour
- month
- dow
- domestic
- arrest (target variable)


In [42]:
df = pd.read_csv("crime_aggregated.csv")

df.head()

,district,primary_type,hour,dow,month,n_incidents,arrest_rate,label_arrest_high
0,9,THEFT,11,7,12,41,0.073171,0
1,14,THEFT,13,6,2,41,0.097561,0
2,3,BATTERY,4,1,12,41,0.243902,0
3,7,NARCOTICS,20,7,2,41,1.000000,1
4,4,NARCOTICS,20,6,1,41,1.000000,1


## 3. Feature Engineering
new columns 
- is_weekend
- is_night
- time_bucket

In [43]:
# weekend indicator
df["is_weekend"] = df["dow"].isin([5,6]).astype(int)

# night crime
df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 4)).astype(int)

# time of day bucket
def time_bucket(h):
    if 5 <= h < 12:
        return "morning"
    elif 12 <= h < 17:
        return "afternoon"
    elif 17 <= h < 22:
        return "evening"
    else:
        return "night"

df["time_bucket"] = df["hour"].apply(time_bucket)


In [44]:
df

,district,primary_type,hour,dow,month,n_incidents,arrest_rate,label_arrest_high,is_weekend,is_night,time_bucket
0,9,THEFT,11,7,12,41,0.073171,0,0,0,morning
1,14,THEFT,13,6,2,41,0.097561,0,1,0,afternoon
2,3,BATTERY,4,1,12,41,0.243902,0,0,1,night
3,7,NARCOTICS,20,7,2,41,1.000000,1,0,0,evening
4,4,NARCOTICS,20,6,1,41,1.000000,1,1,0,evening
...,...,...,...,...,...,...,...,...,...,...,...
133118,8,THEFT,4,7,11,20,0.000000,0,0,1,night
133119,5,THEFT,2,1,11,20,0.050000,0,0,1,night
133120,4,BURGLARY,0,7,11,20,0.000000,0,0,1,night
133121,1,NARCOTICS,6,5,6,20,1.000000,1,1,0,morning


## 4. Define Target and Features

In [45]:
y = df["arrest_rate"]

X = df[["district",
"primary_type",
"hour",
"dow",
"month",
"n_incidents",
"label_arrest_high",
"is_weekend",
"is_night", 
"time_bucket"
]]

## 5. Train/Test Split

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 6. Preprocessing Pipeline

In [47]:
categorical_features = ["primary_type", "time_bucket"]
numeric_features = ["hour","dow","month","n_incidents", "district", "label_arrest_high", "is_weekend", "is_night"]

preprocessor = ColumnTransformer(
[
("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
("num", "passthrough", numeric_features)
]
)

model = Pipeline(
[
("preprocess", preprocessor),
("model", RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
))
]
)

## 7. Model Pipeline

In [48]:
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ))
])

In [ ]:
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 0.059445234913449295
R2: 0.9535473558017892
